## Feature engineering plan

This section defines which variables will be used as model inputs, which variables will be excluded because they represent outcomes, and which new features will be engineered from the raw campaign fields.

#### Feature engineering rationale

Raw campaign fields are useful, but they do not always capture the full business signal on their own. So, we transform raw fields into features that better represent campaign efficiency, timing, and interaction effects. This also supports Responsible AI because the model becomes easier to interpret, easier to audit, and less likely to learn from misleading raw scale differences.

Because this is a Responsible AI decision-support project, the feature set must be chosen carefully. The goal is to build a model that can explain campaign performance without leaking outcome information.

### Core inputs to keep
These describe the campaign setup and context:
- Campaign_Type
- Target_Audience
- Channels_Used
- Duration
- Location
- Language
- Customer_Segment
- Date

### Performance signals to keep
These capture observed campaign behavior:
- Clicks
- Impressions
- Engagement_Score
- Acquisition_Cost

### Fields to exclude from model inputs
These are outcome fields or leakage risks:
- Conversion_Rate
- ROI

### Fields to treat carefully
- Company

### Feature selection reasoning

The model should learn from campaign setup and observed behavior, but not from outcome fields that directly reveal performance. Conversion_Rate and ROI will be used to define and validate the target, but they will not be included as input features. Company is treated cautiously because it may act as a high-cardinality shortcut rather than a generalizable business signal.

#### Why these feature groups matter

- **Time features** help capture seasonality and calendar effects.
- **Efficiency features** normalize campaign behavior so large and small campaigns can be compared fairly.
- **Interaction features** help the model learn that some campaign types work better with certain audiences or channels.
- **Careful exclusion of outcome fields** protects the project from leakage and keeps the model realistic.

In [53]:
import numpy as np
import pandas as pd

def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return numerator / denominator.replace(0, np.nan)

### Feature engineering plan

This section defines the boundary between model inputs, outcome variables, and fields that require caution. The goal is to build a model that is useful for campaign prioritization, explainable to business users, and aligned with Responsible AI principles.

The model should learn from campaign setup and observed campaign behavior, but it should not use outcome fields directly. This keeps the scoring logic realistic and prevents leakage.

In [54]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

DATA_PATH = r'C:\Users\oha14\Projects\AI Campus Cohort 2 -Business Data\marketing_project\data\raw\marketing_campaign_dataset.csv'

campaign_df = pd.read_csv(DATA_PATH)

campaign_work_df = campaign_df.copy()

print("Shape:", campaign_work_df.shape)
display(campaign_work_df.head())

Shape: (200000, 16)


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05


In [55]:
core_inputs = [
    "Campaign_Type",
    "Target_Audience",
    "Channels_Used",
    "Duration",
    "Location",
    "Language",
    "Customer_Segment",
    "Date",
]

performance_signals = [
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "Acquisition_Cost",
]

excluded_columns = [
    "Conversion_Rate",
    "ROI",
]

careful_columns = [
    "Company",
]

feature_boundary = pd.DataFrame({
    "column_group": [
        "Core inputs"] * len(core_inputs)
        + ["Performance signals"] * len(performance_signals)
        + ["Excluded"] * len(excluded_columns)
        + ["Caution"] * len(careful_columns),
    "column_name": core_inputs + performance_signals + excluded_columns + careful_columns,
    "reason": [
        "Describes campaign setup and context"] * len(core_inputs)
        + ["Captures observed campaign behavior"] * len(performance_signals)
        + ["Outcome variable; used for target creation or validation"] * len(excluded_columns)
        + ["High-cardinality field; may act like a shortcut rather than a generalizable signal"] * len(careful_columns),
})

display(feature_boundary)

,column_group,column_name,reason
0,Core inputs,Campaign_Type,Describes campaign setup and context
1,Core inputs,Target_Audience,Describes campaign setup and context
2,Core inputs,Channels_Used,Describes campaign setup and context
3,Core inputs,Duration,Describes campaign setup and context
4,Core inputs,Location,Describes campaign setup and context
5,Core inputs,Language,Describes campaign setup and context
6,Core inputs,Customer_Segment,Describes campaign setup and context
7,Core inputs,Date,Describes campaign setup and context
8,Performance signals,Clicks,Captures observed campaign behavior
9,Performance signals,Impressions,Captures observed campaign behavior


#### Why this boundary matters

The model is designed to support campaign scoring, not to memorize campaign outcomes. Conversion_Rate and ROI are excluded because they describe performance directly and would create leakage if used as features. Company is treated carefully because it may be useful in analysis, but it can also cause the model to learn company-specific shortcuts instead of generalizable patterns.

This boundary is also important for Responsible AI because it keeps the system interpretable, transparent, and tied to business decision-making rather than hidden future information.

## Target definition

The target is defined as a binary label that identifies whether a campaign is high performing. This is more useful for a decision-support project than predicting a raw continuous score because the business needs a ranked list of campaigns, not just a numeric outcome.

A campaign is labeled as high performing if its Conversion_Rate falls in the top quartile of the dataset.

In [56]:
campaign_work_df["Conversion_Rate"] = pd.to_numeric(campaign_work_df["Conversion_Rate"], errors="coerce")

threshold = campaign_work_df["Conversion_Rate"].quantile(0.75)
campaign_work_df["high_performing_campaign"] = (
    campaign_work_df["Conversion_Rate"] >= threshold
).astype(int)

print("High-performance threshold:", threshold)
print("\nTarget counts:")
print(campaign_work_df["high_performing_campaign"].value_counts())

print("\nTarget proportions:")
print(campaign_work_df["high_performing_campaign"].value_counts(normalize=True).round(4))

High-performance threshold: 0.12

Target counts:
high_performing_campaign
0    149984
1     50016
Name: count, dtype: int64

Target proportions:
high_performing_campaign
0    0.7499
1    0.2501
Name: proportion, dtype: float64


### Time-based features

The raw Date column is transformed into business-relevant variables such as year, month, quarter, day of week, and weekend flag. This allows the model to capture timing patterns without using the raw date directly.

Time-based features are useful because campaign performance may vary by season, budget cycle, or audience behavior at different points in the week or year. This is a descriptive transformation, not a causal claim.

In [57]:
campaign_work_df["Date"] = pd.to_datetime(campaign_work_df["Date"], errors="coerce")

campaign_work_df["year"] = campaign_work_df["Date"].dt.year
campaign_work_df["month"] = campaign_work_df["Date"].dt.month
campaign_work_df["month_name"] = campaign_work_df["Date"].dt.month_name()
campaign_work_df["quarter"] = campaign_work_df["Date"].dt.quarter
campaign_work_df["day_of_week"] = campaign_work_df["Date"].dt.day_name()
campaign_work_df["is_weekend"] = campaign_work_df["Date"].dt.dayofweek.isin([5, 6]).astype(int)

display(
    campaign_work_df[["Date", "year", "month", "month_name", "quarter", "day_of_week", "is_weekend"]].head()
)

,Date,year,month,month_name,quarter,day_of_week,is_weekend
0,2021-01-01,2021,1,January,1,Friday,0
1,2021-01-02,2021,1,January,1,Saturday,1
2,2021-01-03,2021,1,January,1,Sunday,1
3,2021-01-04,2021,1,January,1,Monday,0
4,2021-01-05,2021,1,January,1,Tuesday,0


### Efficiency features

Raw counts such as clicks, impressions, engagement, and cost are useful, but they become more informative when converted into rates and per-day measures. These engineered features capture how efficiently a campaign turns visibility into engagement and action.

In [58]:
def clean_numeric_series(series):
    return (
        series.astype(str)
        .str.replace(r"[^0-9.\-]", "", regex=True)
        .replace("", np.nan)
        .astype(float)
    )

campaign_work_df["Duration"] = clean_numeric_series(campaign_work_df["Duration"])
campaign_work_df["Clicks"] = clean_numeric_series(campaign_work_df["Clicks"])
campaign_work_df["Impressions"] = clean_numeric_series(campaign_work_df["Impressions"])
campaign_work_df["Engagement_Score"] = clean_numeric_series(campaign_work_df["Engagement_Score"])
campaign_work_df["Acquisition_Cost"] = clean_numeric_series(campaign_work_df["Acquisition_Cost"])

In [59]:
# Clean Duration: "30 days" -> 30
campaign_work_df["Duration"] = (
    campaign_work_df["Duration"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

# Clean Acquisition_Cost: "$16,174.00" -> 16174.00
campaign_work_df["Acquisition_Cost"] = (
    campaign_work_df["Acquisition_Cost"]
    .astype(str)
    .str.replace(r"[$,]", "", regex=True)
    .astype(float)
)

display(campaign_work_df[["Duration", "Acquisition_Cost"]].head(10))
print(campaign_work_df[["Duration", "Acquisition_Cost"]].isna().sum())

,Duration,Acquisition_Cost
0,30.0,16174.0
1,60.0,11566.0
2,30.0,10200.0
3,60.0,12724.0
4,15.0,16452.0
5,15.0,9716.0
6,60.0,11067.0
7,45.0,13280.0
8,15.0,18066.0
9,15.0,13766.0


Duration            0
Acquisition_Cost    0
dtype: int64


In [60]:
campaign_work_df["click_through_rate"] = campaign_work_df["Clicks"] / campaign_work_df["Impressions"]
campaign_work_df["clicks_per_day"] = campaign_work_df["Clicks"] / campaign_work_df["Duration"]
campaign_work_df["impressions_per_day"] = campaign_work_df["Impressions"] / campaign_work_df["Duration"]
campaign_work_df["engagement_per_day"] = campaign_work_df["Engagement_Score"] / campaign_work_df["Duration"]

campaign_work_df["cost_per_click"] = campaign_work_df["Acquisition_Cost"] / campaign_work_df["Clicks"]
campaign_work_df["cost_per_impression"] = campaign_work_df["Acquisition_Cost"] / campaign_work_df["Impressions"]
campaign_work_df["engagement_efficiency"] = campaign_work_df["Engagement_Score"] / campaign_work_df["Acquisition_Cost"]

campaign_work_df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [61]:
campaign_work_df[
    [
        "Clicks",
        "Impressions",
        "Duration",
        "Engagement_Score",
        "Acquisition_Cost",
        "click_through_rate",
        "clicks_per_day",
        "impressions_per_day",
        "engagement_per_day",
        "cost_per_click",
        "cost_per_impression",
        "engagement_efficiency",
    ]
].head()

,Clicks,Impressions,Duration,Engagement_Score,Acquisition_Cost,click_through_rate,clicks_per_day,impressions_per_day,engagement_per_day,cost_per_click,cost_per_impression,engagement_efficiency
0,506.0,1922.0,30.0,6.0,16174.0,0.263267,16.866667,64.066667,0.200000,31.964427,8.415193,0.000371
1,116.0,7523.0,60.0,7.0,11566.0,0.015419,1.933333,125.383333,0.116667,99.706897,1.537419,0.000605
2,584.0,7698.0,30.0,1.0,10200.0,0.075864,19.466667,256.600000,0.033333,17.465753,1.325019,0.000098
3,217.0,1820.0,60.0,7.0,12724.0,0.119231,3.616667,30.333333,0.116667,58.635945,6.991209,0.000550
4,379.0,4201.0,15.0,3.0,16452.0,0.090217,25.266667,280.066667,0.200000,43.408971,3.916210,0.000182


The raw Duration and Acquisition_Cost fields were successfully converted into numeric values, allowing the campaign efficiency features to be computed correctly. This is an important senior-level step because the project is now using the real business meaning of those fields rather than treating them as unusable text.

This step improves interpretability because the engineered features are easy to explain to business users. Instead of relying on raw counts alone, the model now uses efficiency measures that reflect how well a campaign converts attention and budget into engagement.

### Interaction features

Some campaign variables may not be strongly predictive on their own, but their combinations can reveal useful business patterns. Interaction features help the model learn relationships such as which campaign types work better with which channels, audiences, languages, or customer segments.

This allows the model to capture practical business combinations without making unsupported claims about any single category.

In [62]:
campaign_work_df["campaign_channel_combo"] = (
    campaign_work_df["Campaign_Type"].astype(str).str.strip()
    + "_"
    + campaign_work_df["Channel_Used"].astype(str).str.strip()
)

campaign_work_df["audience_language_combo"] = (
    campaign_work_df["Target_Audience"].astype(str).str.strip()
    + "_"
    + campaign_work_df["Language"].astype(str).str.strip()
)

campaign_work_df["segment_location_combo"] = (
    campaign_work_df["Customer_Segment"].astype(str).str.strip()
    + "_"
    + campaign_work_df["Location"].astype(str).str.strip()
)

campaign_work_df["campaign_segment_combo"] = (
    campaign_work_df["Campaign_Type"].astype(str).str.strip()
    + "_"
    + campaign_work_df["Customer_Segment"].astype(str).str.strip()
)

display(
    campaign_work_df[
        [
            "Campaign_Type",
            "Channel_Used",
            "campaign_channel_combo",
            "Target_Audience",
            "Language",
            "audience_language_combo",
            "Customer_Segment",
            "Location",
            "segment_location_combo",
            "campaign_segment_combo",
        ]
    ].head()
)

,Campaign_Type,Channel_Used,campaign_channel_combo,Target_Audience,Language,audience_language_combo,Customer_Segment,Location,segment_location_combo,campaign_segment_combo
0,Email,Google Ads,Email_Google Ads,Men 18-24,Spanish,Men 18-24_Spanish,Health & Wellness,Chicago,Health & Wellness_Chicago,Email_Health & Wellness
1,Email,Google Ads,Email_Google Ads,Women 35-44,German,Women 35-44_German,Fashionistas,New York,Fashionistas_New York,Email_Fashionistas
2,Influencer,YouTube,Influencer_YouTube,Men 25-34,French,Men 25-34_French,Outdoor Adventurers,Los Angeles,Outdoor Adventurers_Los Angeles,Influencer_Outdoor Adventurers
3,Display,YouTube,Display_YouTube,All Ages,Mandarin,All Ages_Mandarin,Health & Wellness,Miami,Health & Wellness_Miami,Display_Health & Wellness
4,Email,YouTube,Email_YouTube,Men 25-34,Mandarin,Men 25-34_Mandarin,Health & Wellness,Los Angeles,Health & Wellness_Los Angeles,Email_Health & Wellness


### Why this step matters

These interaction features may reveal business structure that is not visible in the single-variable EDA. For example, a campaign type may not look powerful by itself, but it may perform much better when paired with a specific channel or audience. This helps the model learn more realistic campaign patterns and improves the chance of finding high-performing combinations.

From a Responsible AI perspective, these interactions are still interpretable because they are built from business concepts that a human can understand and review.

### Handling 'Company' variable

The Company column is treated cautiously because it is high-cardinality and may behave like a shortcut feature. In a senior-level project, the goal is to learn generalizable campaign patterns, not to memorize company-specific history.

For this first version of the model, Company will be kept for analysis, but excluded from the final model inputs unless later testing shows that it improves generalization in a reliable way.

In [63]:
company_counts = campaign_work_df["Company"].value_counts()

display(company_counts.head(15))
print("Unique companies:", campaign_work_df["Company"].nunique())

Company
TechCorp               40237
Alpha Innovations      40051
DataTech Solutions     40012
NexGen Systems         39991
Innovate Industries    39709
Name: count, dtype: int64

Unique companies: 5


### Why this decision matters

If Company is allowed into the model too early, it may dominate the prediction without teaching the model anything broadly useful about campaign performance. Excluding it from the first model is the safer and more responsible choice because it improves generalization and keeps the system focused on real campaign features.

## Final modeling dataframe

This step combines the selected input features, engineered variables, and the target into one modeling table. The purpose is to create a clean dataset that can be used for training and validation without leaking outcome information.

At this stage, the data should reflect the business logic of the project: campaign setup, observed behavior, efficiency, timing, and meaningful interactions.

In [64]:
feature_columns = [
    "Campaign_Type",
    "Target_Audience",
    "Channel_Used",
    "Duration",
    "Location",
    "Language",
    "Customer_Segment",
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "Acquisition_Cost",
    "year",
    "month",
    "quarter",
    "day_of_week",
    "is_weekend",
    "click_through_rate",
    "clicks_per_day",
    "impressions_per_day",
    "engagement_per_day",
    "cost_per_click",
    "cost_per_impression",
    "engagement_efficiency",
    "campaign_channel_combo",
    "audience_language_combo",
    "segment_location_combo",
    "campaign_segment_combo",
]

target_column = "high_performing_campaign"

modeling_df = campaign_work_df[feature_columns + [target_column]].copy()

print("Modeling dataframe shape:", modeling_df.shape)
display(modeling_df.head())

Modeling dataframe shape: (200000, 28)


,Campaign_Type,Target_Audience,Channel_Used,Duration,Location,Language,Customer_Segment,Clicks,Impressions,Engagement_Score,Acquisition_Cost,year,month,quarter,day_of_week,is_weekend,click_through_rate,clicks_per_day,impressions_per_day,engagement_per_day,cost_per_click,cost_per_impression,engagement_efficiency,campaign_channel_combo,audience_language_combo,segment_location_combo,campaign_segment_combo,high_performing_campaign
0,Email,Men 18-24,Google Ads,30.0,Chicago,Spanish,Health & Wellness,506.0,1922.0,6.0,16174.0,2021,1,1,Friday,0,0.263267,16.866667,64.066667,0.200000,31.964427,8.415193,0.000371,Email_Google Ads,Men 18-24_Spanish,Health & Wellness_Chicago,Email_Health & Wellness,0
1,Email,Women 35-44,Google Ads,60.0,New York,German,Fashionistas,116.0,7523.0,7.0,11566.0,2021,1,1,Saturday,1,0.015419,1.933333,125.383333,0.116667,99.706897,1.537419,0.000605,Email_Google Ads,Women 35-44_German,Fashionistas_New York,Email_Fashionistas,1
2,Influencer,Men 25-34,YouTube,30.0,Los Angeles,French,Outdoor Adventurers,584.0,7698.0,1.0,10200.0,2021,1,1,Sunday,1,0.075864,19.466667,256.600000,0.033333,17.465753,1.325019,0.000098,Influencer_YouTube,Men 25-34_French,Outdoor Adventurers_Los Angeles,Influencer_Outdoor Adventurers,0
3,Display,All Ages,YouTube,60.0,Miami,Mandarin,Health & Wellness,217.0,1820.0,7.0,12724.0,2021,1,1,Monday,0,0.119231,3.616667,30.333333,0.116667,58.635945,6.991209,0.000550,Display_YouTube,All Ages_Mandarin,Health & Wellness_Miami,Display_Health & Wellness,0
4,Email,Men 25-34,YouTube,15.0,Los Angeles,Mandarin,Health & Wellness,379.0,4201.0,3.0,16452.0,2021,1,1,Tuesday,0,0.090217,25.266667,280.066667,0.200000,43.408971,3.916210,0.000182,Email_YouTube,Men 25-34_Mandarin,Health & Wellness_Los Angeles,Email_Health & Wellness,0


### Feature quality check

Before moving to model training, we inspect the engineered features for missing values, infinite values, and basic reasonableness. This step confirms that the transformations worked as intended and that the modeling table is usable.

This matters because a clean-looking feature table can still contain hidden problems after engineering, especially when division-based features are involved.

In [65]:
# Check missing values in the modeling dataframe
feature_missing = pd.DataFrame({
    "missing_count": modeling_df.isna().sum(),
    "missing_pct": (modeling_df.isna().sum() / len(modeling_df) * 100).round(2)
}).sort_values("missing_pct", ascending=False)

display(feature_missing)

print("Total duplicate rows in modeling_df:", modeling_df.duplicated().sum())

,missing_count,missing_pct
Campaign_Type,0,0.0
Target_Audience,0,0.0
campaign_segment_combo,0,0.0
segment_location_combo,0,0.0
audience_language_combo,0,0.0
campaign_channel_combo,0,0.0
engagement_efficiency,0,0.0
cost_per_impression,0,0.0
cost_per_click,0,0.0
engagement_per_day,0,0.0


Total duplicate rows in modeling_df: 0


In [66]:
numeric_checks = [
    "Duration",
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "Acquisition_Cost",
    "click_through_rate",
    "clicks_per_day",
    "impressions_per_day",
    "engagement_per_day",
    "cost_per_click",
    "cost_per_impression",
    "engagement_efficiency",
]

display(modeling_df[numeric_checks].describe().T)

,count,mean,std,min,25%,50%,75%,max
Duration,200000.0,37.503975,16.746720,15.000000,30.000000,30.000000,45.000000,60.000000
Clicks,200000.0,549.772030,260.019056,100.000000,325.000000,550.000000,775.000000,1000.000000
Impressions,200000.0,5507.301520,2596.864286,1000.000000,3266.000000,5517.500000,7753.000000,10000.000000
Engagement_Score,200000.0,5.494710,2.872581,1.000000,3.000000,5.000000,8.000000,10.000000
Acquisition_Cost,200000.0,12504.393040,4337.664545,5000.000000,8739.750000,12496.500000,16264.000000,20000.000000
click_through_rate,200000.0,0.140405,0.130881,0.010054,0.058606,0.099789,0.169699,0.992024
clicks_per_day,200000.0,19.079028,14.840085,1.666667,8.650000,14.666667,24.466667,66.666667
impressions_per_day,200000.0,191.401654,149.054054,16.666667,86.545833,146.886111,244.533333,666.666667
engagement_per_day,200000.0,0.190748,0.156230,0.016667,0.066667,0.133333,0.266667,0.666667
cost_per_click,200000.0,32.008490,26.926121,5.021084,15.091967,22.774008,38.599088,199.960000


### Why this step matters

This check makes sure the model will not be trained on broken engineered fields. If missing values appear here, they usually reflect real division issues or invalid source values that must be handled before modeling. A senior data scientist checks this now rather than discovering it during training.

From a Responsible AI perspective, this step supports transparency and reliability because it prevents hidden preprocessing problems from shaping the model.

### Final feature set for the first model

For the first modeling pass, the project will use campaign context, time features, and engineered efficiency variables. Interaction features and high-cardinality fields are excluded for now so the baseline model stays interpretable and generalizable.

This is a deliberate senior-level choice: start with a clean, defensible feature set, then test more complex interactions only if they improve validation.

In [67]:
feature_columns = [
    "Campaign_Type",
    "Target_Audience",
    "Channel_Used",
    "Duration",
    "Location",
    "Language",
    "Customer_Segment",
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "Acquisition_Cost",
    "year",
    "month",
    "quarter",
    "day_of_week",
    "is_weekend",
    "click_through_rate",
    "clicks_per_day",
    "impressions_per_day",
    "engagement_per_day",
    "cost_per_click",
    "cost_per_impression",
    "engagement_efficiency",
]

target_column = "high_performing_campaign"

modeling_df = campaign_work_df[feature_columns + [target_column]].copy()

print("Modeling dataframe shape:", modeling_df.shape)
display(modeling_df.head())

Modeling dataframe shape: (200000, 24)


,Campaign_Type,Target_Audience,Channel_Used,Duration,Location,Language,Customer_Segment,Clicks,Impressions,Engagement_Score,Acquisition_Cost,year,month,quarter,day_of_week,is_weekend,click_through_rate,clicks_per_day,impressions_per_day,engagement_per_day,cost_per_click,cost_per_impression,engagement_efficiency,high_performing_campaign
0,Email,Men 18-24,Google Ads,30.0,Chicago,Spanish,Health & Wellness,506.0,1922.0,6.0,16174.0,2021,1,1,Friday,0,0.263267,16.866667,64.066667,0.200000,31.964427,8.415193,0.000371,0
1,Email,Women 35-44,Google Ads,60.0,New York,German,Fashionistas,116.0,7523.0,7.0,11566.0,2021,1,1,Saturday,1,0.015419,1.933333,125.383333,0.116667,99.706897,1.537419,0.000605,1
2,Influencer,Men 25-34,YouTube,30.0,Los Angeles,French,Outdoor Adventurers,584.0,7698.0,1.0,10200.0,2021,1,1,Sunday,1,0.075864,19.466667,256.600000,0.033333,17.465753,1.325019,0.000098,0
3,Display,All Ages,YouTube,60.0,Miami,Mandarin,Health & Wellness,217.0,1820.0,7.0,12724.0,2021,1,1,Monday,0,0.119231,3.616667,30.333333,0.116667,58.635945,6.991209,0.000550,0
4,Email,Men 25-34,YouTube,15.0,Los Angeles,Mandarin,Health & Wellness,379.0,4201.0,3.0,16452.0,2021,1,1,Tuesday,0,0.090217,25.266667,280.066667,0.200000,43.408971,3.916210,0.000182,0


### Why these features were selected

These features balance business meaning and predictive value. The raw campaign fields capture setup and context, while the engineered features capture efficiency and relative performance. This gives the model enough structure to learn useful patterns without depending on sparse interaction terms or high-cardinality shortcuts.

The excluded combo features remain available for later experimentation, but the first model should prove that the core campaign signals are strong enough on their own.

In [70]:
campaign_work_df.to_csv("data/processed/campaign_features.csv", index=False)

## Train/test split

The modeling table is now split into training and test sets. This is a crucial step because it lets us evaluate the model on unseen data, which gives a more realistic view of how well it may perform in practice.

Because the target is moderately imbalanced, the split should preserve the class distribution using stratification.

In [68]:
from sklearn.model_selection import train_test_split

X = modeling_df.drop(columns=[target_column])
y = modeling_df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True).round(4))
print("\ny_test distribution:")
print(y_test.value_counts(normalize=True).round(4))

X_train shape: (160000, 23)
X_test shape: (40000, 23)
y_train distribution:
high_performing_campaign
0    0.7499
1    0.2501
Name: proportion, dtype: float64

y_test distribution:
high_performing_campaign
0    0.7499
1    0.2501
Name: proportion, dtype: float64


#### Why this step matters

We do not train on the full dataset and hope the result generalizes. The split creates a fair test of model performance on data the model has not seen before. Stratification matters here because it keeps the target balance consistent across train and test, which is especially important for a prioritization problem.

This also supports Responsible AI, it makes the evaluation more honest and less likely to overstate the model’s usefulness.

### Save split data for modeling

The train and test sets are saved separately so the modeling notebook can reuse them without repeating the feature engineering steps. This makes the project more reproducible and easier to manage inside a notebook workflow.

In [69]:
from pathlib import Path

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_df = X_train.copy()
train_df[target_column] = y_train.values

test_df = X_test.copy()
test_df[target_column] = y_test.values

train_df.to_csv(PROCESSED_DIR / "train_data.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_data.csv", index=False)

print("Saved:", PROCESSED_DIR / "train_data.csv")
print("Saved:", PROCESSED_DIR / "test_data.csv")

Saved: data\processed\train_data.csv
Saved: data\processed\test_data.csv
